In [1]:
from typing import List, Set, Optional
import io
import sys

# B树节点类，小幅调整命名区分原代码
class BTreeUnit:
    def __init__(self, is_leaf: bool = False):
        self.key_list: List[int] = []
        self.child_list: List[BTreeUnit] = []
        self.is_leaf_node = is_leaf

# B树主体类
class MultiWayTree:
    def __init__(self, min_degree: int):
        self.root_node = BTreeUnit(is_leaf=True)
        self.min_deg = min_degree
        self.max_key_num = 2 * min_degree - 1
        self.min_key_num = min_degree - 1

    def add_value(self, num: int):
        root = self.root_node
        if len(root.key_list) == self.max_key_num:
            new_top = BTreeUnit(is_leaf=False)
            new_top.child_list.append(root)
            self.split_child_node(new_top, 0)
            self.root_node = new_top
            self.insert_not_full(new_top, num)
        else:
            self.insert_not_full(root, num)

    def split_child_node(self, parent_node: BTreeUnit, child_index: int):
        t = self.min_deg
        full_child = parent_node.child_list[child_index]
        new_sibling = BTreeUnit(full_child.is_leaf_node)
        mid_val = full_child.key_list[t - 1]

        # 拆分键值
        new_sibling.key_list = full_child.key_list[t:]
        full_child.key_list = full_child.key_list[:t - 1]

        # 拆分子节点
        if not full_child.is_leaf_node:
            new_sibling.child_list = full_child.child_list[t:]
            full_child.child_list = full_child.child_list[:t]

        parent_node.key_list.insert(child_index, mid_val)
        parent_node.child_list.insert(child_index + 1, new_sibling)

    def insert_not_full(self, node: BTreeUnit, target: int):
        ptr = len(node.key_list) - 1
        if node.is_leaf_node:
            node.key_list.append(target)
            node.key_list.sort()
        else:
            while ptr >= 0 and target < node.key_list[ptr]:
                ptr -= 1
            ptr += 1
            if len(node.child_list[ptr].key_list) == self.max_key_num:
                self.split_child_node(node, ptr)
                if target > node.key_list[ptr]:
                    ptr += 1
            self.insert_not_full(node.child_list[ptr], target)

    def show_tree_struct(self, node: Optional[BTreeUnit] = None, depth: int = 0):
        if node is None:
            node = self.root_node
        blank = "  " * depth
        key_text = "、".join(map(str, node.key_list))
        node_type = "叶子节点" if node.is_leaf_node else "内部节点"
        child_count = len(node.child_list)
        print(f"{blank}[{key_text}] 子节点数:{child_count} | {node_type}")
        if not node.is_leaf_node:
            for kid in node.child_list:
                self.show_tree_struct(kid, depth + 1)

    def check_all_rules(self) -> bool:
        print("\nB树约束条件校验结果")
        print("-" * 52)
        leaf_layer_record = []
        self._record_leaf_layer(self.root_node, 0, leaf_layer_record)
        all_same_layer = len(set(leaf_layer_record)) == 1
        print(f"1. 全部叶子节点处于同一层级：{'√' if all_same_layer else '×'}")

        max_k = self.max_key_num
        key_count_ok = self._check_max_key_limit(self.root_node, max_k)
        print(f"2. 单节点键值不超过{max_k}个：{'√' if key_count_ok else '×'}")

        root_key_ok = len(self.root_node.key_list) >= 1
        print(f"3. 根节点至少包含1个键值：{'√' if root_key_ok else '×'}")

        min_k = self.min_key_num
        non_root_key_ok = self._check_min_key_limit(self.root_node, min_k, True)
        print(f"4. 非根节点至少包含{min_k}个键值：{'√' if non_root_key_ok else '×'}")

        sort_ok = self._check_key_order(self.root_node)
        print(f"5. 节点内键值升序排列：{'√' if sort_ok else '×'}")

        child_match_ok = self._check_child_count_match(self.root_node)
        print(f"6. 内部节点子节点数量 = 键值数+1：{'√' if child_match_ok else '×'}")

        unique_ok = self._check_all_unique(self.root_node)
        print(f"7. 树内所有键值互不重复：{'√' if unique_ok else '×'}")

        separate_ok = self._check_separate_rule(self.root_node)
        print(f"8. 子树键值分隔大小规则符合：{'√' if separate_ok else '×'}")

        all_check = [all_same_layer, key_count_ok, root_key_ok, non_root_key_ok,
                     sort_ok, child_match_ok, unique_ok, separate_ok]
        return all(all_check)

    def _record_leaf_layer(self, node: BTreeUnit, layer: int, record: List[int]):
        if node.is_leaf_node:
            record.append(layer)
        else:
            for child in node.child_list:
                self._record_leaf_layer(child, layer + 1, record)

    def _check_max_key_limit(self, node: BTreeUnit, upper: int) -> bool:
        if len(node.key_list) > upper:
            return False
        if not node.is_leaf_node:
            for c in node.child_list:
                if not self._check_max_key_limit(c, upper):
                    return False
        return True

    def _check_min_key_limit(self, node: BTreeUnit, lower: int, is_root: bool) -> bool:
        if not is_root and len(node.key_list) < lower:
            return False
        if not node.is_leaf_node:
            for c in node.child_list:
                if not self._check_min_key_limit(c, lower, False):
                    return False
        return True

    def _check_key_order(self, node: BTreeUnit) -> bool:
        for idx in range(1, len(node.key_list)):
            if node.key_list[idx] <= node.key_list[idx - 1]:
                return False
        if not node.is_leaf_node:
            for c in node.child_list:
                if not self._check_key_order(c):
                    return False
        return True

    def _check_child_count_match(self, node: BTreeUnit) -> bool:
        if not node.is_leaf_node:
            expect = len(node.key_list) + 1
            if len(node.child_list) != expect:
                return False
            for c in node.child_list:
                if not self._check_child_count_match(c):
                    return False
        return True

    def _check_all_unique(self, node: BTreeUnit, exist_set: Set[int] = None) -> bool:
        if exist_set is None:
            exist_set = set()
        for num in node.key_list:
            if num in exist_set:
                return False
            exist_set.add(num)
        if not node.is_leaf_node:
            for c in node.child_list:
                if not self._check_all_unique(c, exist_set):
                    return False
        return True

    def _fetch_all_sub_keys(self, node: BTreeUnit) -> List[int]:
        res = node.key_list.copy()
        if not node.is_leaf_node:
            for c in node.child_list:
                res.extend(self._fetch_all_sub_keys(c))
        return res

    def _check_separate_rule(self, node: BTreeUnit) -> bool:
        if node.is_leaf_node:
            return True
        for idx in range(len(node.key_list)):
            mid_val = node.key_list[idx]
            left_sub = self._fetch_all_sub_keys(node.child_list[idx])
            if any(x >= mid_val for x in left_sub):
                return False
            if idx + 1 < len(node.child_list):
                right_sub = self._fetch_all_sub_keys(node.child_list[idx + 1])
                if any(x <= mid_val for x in right_sub):
                    return False
        for c in node.child_list:
            if not self._check_separate_rule(c):
                return False
        return True

# 主运行入口
if __name__ == "__main__":
    print("初始化3阶B树（最小度数t=2）")
    insert_seq = [10, 20, 5, 6, 12, 30, 25]
    print(f"待插入数值序列：{insert_seq}")
    print("-" * 60)

    tree = MultiWayTree(min_degree=2)
    for step, data in enumerate(insert_seq):
        print(f"\n第{step + 1}轮插入数值 {data}")
        tree.add_value(data)
        print("当前树结构展示：")
        tree.show_tree_struct()

    print("\n" + "-" * 60)
    print("插入完成后的完整B树结构")
    tree.show_tree_struct()

    check_result = tree.check_all_rules()

    print("\n" + "-" * 60)
    tip = "全部校验通过" if check_result else "存在不满足约束的结构"
    print(f"B树全部规则校验：{tip}")

    # 导出报告文件
    with open("btree_analysis_report.txt", "w", encoding="utf-8") as file:
        file.write("3阶B树构建分析报告\n")
        file.write("-" * 50 + "\n\n")
        file.write(f"插入数据序列：{insert_seq}\n\n")
        file.write("最终树形结构：\n")

        # 捕获打印输出写入文件
        stdout_cache = sys.stdout
        buffer = io.StringIO()
        sys.stdout = buffer
        tree.show_tree_struct()
        sys.stdout = stdout_cache
        tree_text = buffer.getvalue()
        file.write(tree_text)

        file.write("\n约束校验总结：\n")
        file.write(f"整体校验状态：{tip}\n")

    print("\n文件 btree_analysis_report.txt 已生成完毕")

初始化3阶B树（最小度数t=2）
待插入数值序列：[10, 20, 5, 6, 12, 30, 25]
------------------------------------------------------------

第1轮插入数值 10
当前树结构展示：
[10] 子节点数:0 | 叶子节点

第2轮插入数值 20
当前树结构展示：
[10、20] 子节点数:0 | 叶子节点

第3轮插入数值 5
当前树结构展示：
[5、10、20] 子节点数:0 | 叶子节点

第4轮插入数值 6
当前树结构展示：
[10] 子节点数:2 | 内部节点
  [5、6] 子节点数:0 | 叶子节点
  [20] 子节点数:0 | 叶子节点

第5轮插入数值 12
当前树结构展示：
[10] 子节点数:2 | 内部节点
  [5、6] 子节点数:0 | 叶子节点
  [12、20] 子节点数:0 | 叶子节点

第6轮插入数值 30
当前树结构展示：
[10] 子节点数:2 | 内部节点
  [5、6] 子节点数:0 | 叶子节点
  [12、20、30] 子节点数:0 | 叶子节点

第7轮插入数值 25
当前树结构展示：
[10、20] 子节点数:3 | 内部节点
  [5、6] 子节点数:0 | 叶子节点
  [12] 子节点数:0 | 叶子节点
  [25、30] 子节点数:0 | 叶子节点

------------------------------------------------------------
插入完成后的完整B树结构
[10、20] 子节点数:3 | 内部节点
  [5、6] 子节点数:0 | 叶子节点
  [12] 子节点数:0 | 叶子节点
  [25、30] 子节点数:0 | 叶子节点

B树约束条件校验结果
----------------------------------------------------
1. 全部叶子节点处于同一层级：√
2. 单节点键值不超过3个：√
3. 根节点至少包含1个键值：√
4. 非根节点至少包含1个键值：√
5. 节点内键值升序排列：√
6. 内部节点子节点数量 = 键值数+1：√
7. 树内所有键值互不重复：√
8. 子树键值分隔大小规则符合：√

----------------------